# Phase 14A — SQL Generator Fine-tuning (Qwen2.5-Coder-7B-Instruct)

Fine-tunes `Qwen2.5-Coder-7B-Instruct` via LoRA to generate SQL queries
given the full pipeline context: enriched schema + key fields + SAR examples.

## What this notebook does

1. **Build training data** — for each of the 6748 CoT entries:
   - Retrieves top-3 structurally similar Q-SQL pairs from ChromaDB (SAR)
   - Formats a prompt: schema → key_fields → 3 SAR examples → question
   - Label: ground-truth SQL
2. **Fine-tune** — LoRA SFT on Qwen2.5-Coder-7B-Instruct (3 epochs)
3. **Save** checkpoint to Google Drive

## Prompt format (what the model learns)

```
System: You are an expert SQL query writer...

User:
## Database Schema
# Table: singer
[(singer_id:INT, ...), (name:TEXT, Examples: [Ed Sheeran, Adele]), ...]

## Key Fields
singer.country, singer.age

## Similar Examples
Example 1:
Q: How many singers are older than 25?
SQL: SELECT COUNT(*) FROM singer WHERE age > 25

## Question
How many singers are from France?

Assistant: SELECT COUNT(*) FROM singer WHERE country = 'France'
```

## Prerequisites
- Phase 12A ✅: `sar_sql/sar_model.pt` on Drive
- Phase 13 ✅: `indexes/chroma_sql/` on Drive
- Phase 8A ✅: `sql_cot_train.json` in repo

> **Requires A100** for efficient training. T4 works but is slow (~8-10 hrs).
> Runtime → Change runtime type → A100.

## Cell 1 — Mount Drive and set paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE     = '/content/drive/MyDrive/codegen'
SAR_MODEL      = f'{DRIVE_BASE}/checkpoints/sar_sql/sar_model.pt'
CHROMA_SQL_DIR = f'{DRIVE_BASE}/indexes/chroma_sql'
GEN_OUT_DIR    = f'{DRIVE_BASE}/checkpoints/generator_sql'
GEN_DATA_DIR   = f'{DRIVE_BASE}/generator_data'
GEN_DATA_FILE  = f'{GEN_DATA_DIR}/sql_generator_train.jsonl'

os.makedirs(GEN_OUT_DIR,  exist_ok=True)
os.makedirs(GEN_DATA_DIR, exist_ok=True)

# Verify prerequisites
for path, label in [
    (SAR_MODEL,                    'SAR SQL model'),
    (CHROMA_SQL_DIR,               'ChromaDB SQL index'),
]:
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists and os.path.isfile(path) else ''
    print(f'{label}: {"✅" if exists else "❌"} {size}')

import torch
print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
USE_A100 = torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(0)
print(f'A100 mode: {USE_A100}')

## Cell 2 — Clone / update repo

In [ ]:
%%bash
set -e
REPO="/content/Codegen"
BRANCH="phase/14a-generator-sql"

if [ -d "$REPO/.git" ]; then
    cd "$REPO" && git fetch origin && git checkout $BRANCH && git pull origin $BRANCH
else
    git clone https://github.com/kethansplunk/Codegen.git "$REPO"
    cd "$REPO" && git checkout $BRANCH
fi
echo "Branch : $(git branch --show-current)"
echo "Commit : $(git log --oneline -1)"

## Cell 3 — Install dependencies

In [ ]:
!pip install -q chromadb "FlagEmbedding==1.2.9" trl peft bitsandbytes accelerate
print('Dependencies installed ✅')

## Cell 4 — Build SQL generator training data

For each of the 6748 CoT training entries:
1. Query ChromaDB for top-3 structurally similar SQL examples (SAR)
2. Format: schema + key_fields + 3 SAR examples + question → SQL
3. Save to `sql_generator_train.jsonl` on Drive

**Skips if already built** (resumes from checkpoint if interrupted).

Expected time: ~15-20 minutes (BGE encodes each question once)

In [ ]:
import os, sys
sys.path.insert(0, '/content/Codegen')

# Check if already built
if os.path.exists(GEN_DATA_FILE):
    with open(GEN_DATA_FILE) as f:
        n = sum(1 for _ in f)
    print(f'Training data already exists: {n} entries — skipping build ✅')
else:
    import subprocess, shlex
    cmd = (
        f'python -m scripts.build_generator_training_data '
        f'--cot        /content/Codegen/Data/cot_data/sql_cot_train.json '
        f'--chroma_dir {CHROMA_SQL_DIR} '
        f'--sar_model  {SAR_MODEL} '
        f'--out        {GEN_DATA_FILE}'
    )
    result = subprocess.run(shlex.split(cmd), cwd='/content/Codegen',
                            capture_output=False, text=True)
    if result.returncode != 0:
        raise RuntimeError('Training data build failed')

## Cell 5 — Inspect training data

Verify the format before training.

In [ ]:
import json

with open(GEN_DATA_FILE) as f:
    samples = [json.loads(line) for line in f]

print(f'Training examples: {len(samples)}')
print(f'\n=== Sample entry (truncated) ===')
print(samples[0]['text'][:1200])
print('...')

## Cell 6 — Fine-tune SQL Generator

**T4**: 4-bit QLoRA, batch=1, grad_accum=16, max_len=1024 (~8-10 hours for 3 epochs)

**A100**: bf16 LoRA, batch=4, grad_accum=4, max_len=2048 (~2-3 hours for 3 epochs)

Only the SQL tokens (after `<|im_start|>assistant\n`) receive loss — the prompt is masked.

Checkpoint saved after each epoch to Drive at `checkpoints/generator_sql/`.

In [ ]:
%%bash
cd /content/Codegen

# Symlink training data from Drive into repo Data/ dir
mkdir -p Data/generator_data
ln -sf "$GEN_DATA_FILE" Data/generator_data/sql_generator_train.jsonl 2>/dev/null || true

python -m src.generator.train \
    --data  "$GEN_DATA_FILE" \
    --out   "$GEN_OUT_DIR" \
    $([ "$USE_A100" = "True" ] && echo "--use_a100" || echo "")

## Cell 7 — Verify saved checkpoint

In [ ]:
import os

print(f'Checkpoint directory: {GEN_OUT_DIR}')
files = os.listdir(GEN_OUT_DIR) if os.path.exists(GEN_OUT_DIR) else []
total_mb = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fs in os.walk(GEN_OUT_DIR)
    for f in fs
) / 1e6 if files else 0

print(f'Files  : {len(files)}')
print(f'Size   : {total_mb:.1f} MB')
for fname in sorted(files):
    print(f'  {fname}')

## Cell 8 — Smoke test: generate SQL for a sample question

In [ ]:
import sys
sys.path.insert(0, '/content/Codegen')

from src.generator.infer import GeneratorInfer
from src.sar.infer import ChromaSARRetriever

# Load retriever
retriever = ChromaSARRetriever(
    model_path=SAR_MODEL,
    chroma_dir=CHROMA_SQL_DIR,
    collection_name='sar_sql',
)

# Load generator
gen = GeneratorInfer(GEN_OUT_DIR, n_candidates=1, temperature=0.0)

# Test
question   = 'How many singers are there in each country?'
schema     = '# Table: singer\n[(singer_id:INT), (name:TEXT), (country:TEXT), (age:INT)]'
key_fields = ['singer.country']
examples   = retriever.retrieve(question, top_k=3)

sql = gen.generate(question, schema, key_fields, examples)[0]
print(f'Question : {question}')
print(f'SQL      : {sql}')